# ONS daily peak demand: EDA, 7-day XGBoost and 2027 forecast (CSV ingestion)

This notebook reads the **CSV** files published by ONS. Its twin, `01_parquet.ipynb`, reads the Parquet files.
Only the ingestion SQL (`sql/00_raw_csv.sql`) differs. Staging, features and models live in `sql/` and `src/pipeline.py` and are shared.

**Target:** `val_demandainstantanea`, the daily maximum instantaneous demand per subsystem (MW).

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pipeline as p

FORMAT = "csv"
FIG = ROOT / "docs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "grid.alpha": 0.3})
SUBS = ["SE", "S", "NE", "N"]
NAMES = {"SE": "Southeast/Midwest", "S": "South", "NE": "Northeast", "N": "North"}

## 1. Ingestion

The raw view, as the file delivers it, before any typing:

In [ ]:
print((ROOT / "sql" / "00_raw_csv.sql").read_text())
con, ingest_s = p.connect(FORMAT)
files = sorted((ROOT / "data" / "raw").glob("*.csv"))
size_kb = sum(f.stat().st_size for f in files) / 1024
print(f"{len(files)} files, {size_kb:,.0f} KB on disk, ingestion {ingest_s*1000:,.0f} ms")
con.sql("DESCRIBE raw_demanda_maxima").df()[["column_name", "column_type"]]

## 2. Staging and data quality (SQL)

In [ ]:
con.sql("SELECT * FROM stg_demanda_maxima ORDER BY din_instante, id_subsistema LIMIT 8").df()

In [ ]:
con.sql('''
SELECT id_subsistema,
       COUNT(*)                                   AS days,
       MIN(din_instante)                          AS first_day,
       MAX(din_instante)                          AS last_day,
       COUNT(*) FILTER (demanda_maxima_mw IS NULL) AS missing_days,
       ROUND(MIN(demanda_maxima_mw))              AS min_mw,
       ROUND(AVG(demanda_maxima_mw))              AS mean_mw,
       ROUND(MAX(demanda_maxima_mw))              AS max_mw
FROM fct_demanda_diaria
GROUP BY 1 ORDER BY mean_mw DESC
''').df()

In [ ]:
dupes = con.sql("SELECT din_instante, id_subsistema, COUNT(*) n FROM stg_demanda_maxima GROUP BY 1, 2 HAVING n > 1").df()
inst_lt_int = con.sql("SELECT COUNT(*) AS n FROM stg_demanda_maxima WHERE demanda_maxima_mw < demanda_integralizada_mwmed").df().n[0]
print("duplicate (day, subsystem) rows:", len(dupes))
print("days where instantaneous peak < integrated value:", inst_lt_int)

## 3. Exploratory analysis

In [ ]:
df = con.sql("SELECT * FROM fct_demanda_diaria").df()
fig, axes = plt.subplots(4, 1, figsize=(12, 11), sharex=True)
for ax, s in zip(axes, SUBS):
    g = df[df.id_subsistema == s].sort_values("din_instante")
    ax.plot(g.din_instante, g.demanda_maxima_mw, lw=0.6, alpha=0.5, label="daily peak")
    ax.plot(g.din_instante, g.demanda_maxima_mw.rolling(28, center=True).mean(), lw=1.8, label="28-day mean")
    ax.set_title(NAMES[s]); ax.set_ylabel("MW")
axes[0].legend(loc="upper left")
fig.tight_layout(); fig.savefig(FIG / "01_daily_peak.png", dpi=120); plt.show()

Southeast/Midwest is about 3x any other subsystem, so errors are compared with **WAPE** and relative skill, not absolute MW.
Summer (Jan to Mar) peaks are clear in SE and S, driven by air conditioning. North and Northeast grow fastest (5 to 8% a year); SE grows about 2%.

In [ ]:
annual = con.sql('''
SELECT id_subsistema, YEAR(din_instante) AS year, ROUND(AVG(demanda_maxima_mw)) AS mean_peak_mw
FROM fct_demanda_diaria
WHERE MONTH(din_instante) <= 10          -- Jan-Oct only: 2026 is incomplete
GROUP BY 1, 2 ORDER BY 1, 2
''').df().pivot(index="id_subsistema", columns="year", values="mean_peak_mw")
annual["growth_25_vs_24"] = annual[2025] / annual[2024] - 1
annual["growth_26_vs_25"] = annual[2026] / annual[2025] - 1
annual.style.format({2024: "{:,.0f}", 2025: "{:,.0f}", 2026: "{:,.0f}", "growth_25_vs_24": "{:+.1%}", "growth_26_vs_25": "{:+.1%}"})

In [ ]:
feat = con.sql("SELECT * FROM fct_demanda_features").df()
feat["norm"] = feat.demanda_maxima_mw / feat.groupby("id_subsistema").demanda_maxima_mw.transform("mean")
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
for s in SUBS:
    g = feat[feat.id_subsistema == s]
    a1.plot(days, g.groupby("dia_semana").norm.mean(), marker="o", label=NAMES[s])
    a2.plot(range(1, 13), g.groupby("mes").norm.mean(), marker="o", label=NAMES[s])
a1.set_title("Weekly profile (peak / subsystem mean)"); a2.set_title("Monthly profile (peak / subsystem mean)")
a2.set_xticks(range(1, 13)); a1.legend()
fig.tight_layout(); fig.savefig(FIG / "02_weekly_monthly_profile.png", dpi=120); plt.show()

Sundays sit 11% (SE) to 18% (S) below midweek; in N and NE only about 5%. The weekly cycle is the reason seasonal naive (same weekday last week) is the right floor.

In [ ]:
hours = df.assign(hour=df.din_pico.dt.hour)
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5), sharey=True)
for ax, s in zip(axes, SUBS):
    h = hours[hours.id_subsistema == s].groupby("hour").size().reindex(range(24), fill_value=0)
    ax.bar(h.index, h / h.sum()); ax.set_title(NAMES[s]); ax.set_xlabel("hour of peak")
axes[0].set_ylabel("share of days")
fig.tight_layout(); fig.savefig(FIG / "03_peak_hour.png", dpi=120); plt.show()

SE and S peak mostly at 18h to 19h, NE at 21h, and N is split between late night and early afternoon.
Few afternoon peaks in SE and S is consistent with behind-the-meter solar (MMGD) shaving the midday load: the system peak arrives as solar output fades.

In [ ]:
hol = con.sql('''
SELECT f.nome AS holiday, s.id_subsistema,
       s.demanda_maxima_mw / s.lag_7d - 1 AS vs_same_weekday_last_week
FROM fct_demanda_features s JOIN feriados f ON f.data = s.din_instante
WHERE s.lag_7d IS NOT NULL AND s.dia_semana <= 5
''').df()
(hol.groupby("holiday").vs_same_weekday_last_week.mean().sort_values()
    .plot.barh(figsize=(8, 5), title="Weekday holidays: peak vs same weekday previous week"))
plt.gca().xaxis.set_major_formatter(lambda x, _: f"{x:+.0%}")
plt.tight_layout(); plt.savefig(FIG / "04_holiday_effect.png", dpi=120); plt.show()

Weekday holidays cut the peak by up to 13% (Christmas); Independence Day, Good Friday and Carnival by 7 to 10%. This is why `feriado` is a feature and not left to the model to guess.

## 4. 7-day-ahead model: XGBoost vs seasonal naive

Features come from `sql/04_fct_demanda_features.sql`. All of them are at least 7 days old, so nothing from the forecast week leaks in.
Validation is walk-forward with an expanding window. The holdout (2026-08-15 to 2026-11-14) is used once, below.

In [ ]:
print((ROOT / "sql" / "04_fct_demanda_features.sql").read_text())

In [ ]:
cv = p.walk_forward(feat)
cv.pivot(index="test", columns="model", values="WAPE").style.format("{:.2%}")

In [ ]:
cv.groupby("model")[["WAPE", "MAPE", "skill_vs_naive"]].mean().style.format("{:.2%}")

### Holdout (touched once)

In [ ]:
ho = p.holdout(feat)
models = ["seasonal_naive", "last_week_mean", "xgboost"]
display(p.scores(ho, models).style.format("{:.2%}"))
pd.concat({NAMES[s]: p.scores(g, models)["WAPE"] for s, g in ho.groupby("id_subsistema")}, axis=1).T.style.format("{:.2%}")

In [ ]:
g = ho[ho.id_subsistema == "SE"]
plt.plot(g.din_instante, g.demanda_maxima_mw, "k", lw=2, label="actual")
plt.plot(g.din_instante, g.seasonal_naive, "--", label="seasonal naive")
plt.plot(g.din_instante, g.xgboost, label="XGBoost")
plt.title("Southeast/Midwest, holdout, 7 days ahead"); plt.ylabel("MW"); plt.legend()
plt.tight_layout(); plt.savefig(FIG / "05_holdout_se.png", dpi=120); plt.show()

In [ ]:
from xgboost import XGBRegressor
train = p.add_baselines(feat.dropna(subset=["lag_7d", "media_movel_7d"]))
m = XGBRegressor(**p.XGB_PARAMS).fit(train[p.FEATURES_7D], train.demanda_maxima_mw)
pd.Series(m.feature_importances_, p.FEATURES_7D).sort_values().plot.barh(figsize=(7, 4), title="XGBoost feature importance (gain)")
plt.tight_layout(); plt.savefig(FIG / "06_feature_importance.png", dpi=120); plt.show()

## 5. 2027 forecast (long horizon)

Lag features don't exist a year ahead, so this is a different model: a **linear trend per subsystem x an XGBoost seasonal ratio** using calendar features only (subsystem, weekday, month, day of year, holiday).
XGBoost alone cannot extrapolate growth, because trees predict inside the range they were trained on. The trend carries the growth.

First, a backtest of the method: train up to 2025-11-14, forecast the next 365 days, and compare with the annual naive (same weekday, 52 weeks earlier).

In [ ]:
tr = feat[feat.din_instante < "2025-11-15"]
te = feat[feat.din_instante >= "2025-11-15"].copy()
te["annual_model"] = p.AnnualModel().fit(tr).predict(te)
te["seasonal_naive"] = te["lag_52w"]
display(p.scores(te, ["seasonal_naive", "annual_model"]).style.format("{:.2%}"))
pd.concat({NAMES[s]: p.scores(g, ["seasonal_naive", "annual_model"])["WAPE"] for s, g in te.groupby("id_subsistema")}, axis=1).T.style.format("{:.2%}")

In [ ]:
model = p.AnnualModel().fit(feat)
fc = p.calendar_2027(con)
fc["forecast_mw"] = model.predict(fc).round(1)
(ROOT / "outputs").mkdir(exist_ok=True)
fc[["din_instante", "id_subsistema", "nom_subsistema", "feriado", "forecast_mw"]].to_csv(ROOT / "outputs" / "forecast_2027.csv", index=False)

fig, axes = plt.subplots(4, 1, figsize=(12, 11), sharex=True)
for ax, s in zip(axes, SUBS):
    h = feat[feat.id_subsistema == s].sort_values("din_instante"); f = fc[fc.id_subsistema == s]
    ax.plot(h.din_instante, h.demanda_maxima_mw, lw=0.6, color="gray", label="history")
    ax.plot(f.din_instante, f.forecast_mw, lw=0.8, color="C1", label="2027 forecast")
    ax.set_title(NAMES[s]); ax.set_ylabel("MW")
axes[0].legend(loc="upper left")
fig.tight_layout(); fig.savefig(FIG / "07_forecast_2027.png", dpi=120); plt.show()

In [ ]:
monthly = fc.groupby(["id_subsistema", "mes"]).forecast_mw.max().unstack(0)[SUBS].rename(columns=NAMES)
monthly.index.name = "2027 month"
monthly.style.format("{:,.0f}").set_caption("Forecast monthly maximum of daily peak, MW")

## 6. Format note (CSV)

Size on disk and ingestion time for this format are printed in section 1. CSV arrives as text (all `VARCHAR`), so every date and number is cast in staging, and padded codes like `"N  "` need `TRIM`.
`python src/pipeline.py` asserts that both formats produce an identical feature table.